# Análise exploratória dos dados




### Entendendo o dataset

In [2]:
# Importações de depenências

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Carregando o dataset
df = pd.read_excel('../data/raw/churn.xlsx')

# Informações básicas do dataset
print("=== ESTATÍSTICAS BÁSICAS ===")
print(f"Total de clientes: {len(df)}")
print(f"\nPrimeiras 5 linhas:")
print(df.head(5))



=== ESTATÍSTICAS BÁSICAS ===
Total de clientes: 7043

Primeiras 5 linhas:
   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   

                 Lat Long   Latitude   Longitude  Gender  ...        Contract  \
0  33.964131, -118.272783  33.964131 -118.272783    Male  ...  Month-to-month   
1   34.059281, -118.30742  34.059281 -118.307420  Female  ...  Month-to-month   
2  34.048013, -118.293953  34.048013 -118.293953  Female  ...  Month-to-month   
3  34.062125, -118.315709  34.062125 -118.315709  Female  ...  Month-to-month   
4  34.039224, -118.266293  34.039224 -118.266293    Male  ...  Month-to-mont

In [3]:
print("\n=== ESTATÍSTICAS DO DATASET ===")
print(df.describe())


=== ESTATÍSTICAS DO DATASET ===
        Count      Zip Code     Latitude    Longitude  Tenure Months  \
count  7043.0   7043.000000  7043.000000  7043.000000    7043.000000   
mean      1.0  93521.964646    36.282441  -119.798880      32.371149   
std       0.0   1865.794555     2.455723     2.157889      24.559481   
min       1.0  90001.000000    32.555828  -124.301372       0.000000   
25%       1.0  92102.000000    34.030915  -121.815412       9.000000   
50%       1.0  93552.000000    36.391777  -119.730885      29.000000   
75%       1.0  95351.000000    38.224869  -118.043237      55.000000   
max       1.0  96161.000000    41.962127  -114.192901      72.000000   

       Monthly Charges  Churn Value  Churn Score         CLTV  
count      7043.000000  7043.000000  7043.000000  7043.000000  
mean         64.761692     0.265370    58.699418  4400.295755  
std          30.090047     0.441561    21.525131  1183.057152  
min          18.250000     0.000000     5.000000  2003.000000 

In [ ]:
# Informações sobre o dataset
print("\n=== INFORMAÇÕES DO DATASET ===")
print(df.info())

### Entendendo o alvo

In [ ]:
# Entendendo o alvo
colunas_sobre_churn = ['Churn Label', 'Churn Value', 'Churn Score']

print("=== COLUNAS SOBRE CHURN ===")
for i in colunas_sobre_churn:
    print(f"{i}:")
    print(df[i].value_counts())
    print("\n")

In [ ]:
# Entendendo colunas 

print(df['Count'].value_counts())

In [ ]:
# Entendendo a variável alvo
target = df[df['Churn Value'] == 1]

pd.options.display.max_columns = None
target.head(2)

### Limpeza dos dados

In [ ]:
# Primeira limpeza de dados

df = df.drop(['Churn Label', 'Churn Score', 'Churn Reason', 'Country', 'State', 'CustomerID', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Count'], axis=1)

In [ ]:

# Força a conversão para numérico. O que não for número vira NaN.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# (Opcional, mas recomendado) Substituir os NaNs por 0, já que o cliente ainda não pagou nada
df['Total Charges'] = df['Total Charges'].fillna(0)

# Verifique o tipo agora. Deve retornar float64!
print("Tipo da coluna:", df['Total Charges'].dtype)

### Analisando correlações das variáveis numéricas

In [ ]:
# Verificando correlação entre variáveis numéricas e target

import seaborn as sns
import matplotlib.pyplot as plt

# Seleciona apenas as colunas numéricas 
colunas_numericas = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV', 'Churn Value']
df_numerico = df[colunas_numericas]

# Calcula a matriz de correlação
matriz_corr = df_numerico.corr()

# Plota o heatmap (Mapa de Calor)
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlação de Variáveis Numéricas com Churn')
plt.show()




### Analisando correlações das variáveis categóricas

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Lista das colunas categóricas que queremos investigar
colunas_categoricas = ['Senior Citizen', 'Partner', 'Dependents', 'Gender', 'Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method' ]

# Configura o visual do Seaborn (deixa os gráficos mais bonitos)
sns.set_theme(style="whitegrid")

# Loop para plotar um gráfico por coluna
for coluna in colunas_categoricas:
    plt.figure(figsize=(8, 4))
    
    # O barplot com y='Churn Value' calcula automaticamente a média (taxa de churn)
    sns.barplot(x=coluna, y='Churn Value', data=df, errorbar=None, palette='viridis')
    
    plt.title(f'Taxa de Churn por {coluna}')
    plt.ylabel('Taxa de Churn')
    plt.xlabel(coluna)
    
    # Define o limite do eixo Y de 0 a 1 (0% a 100%) para facilitar a leitura
    plt.ylim(0, 1) 
    
    plt.show()

### Analisando correlações da coluna "City" e Churn

In [ ]:
# Histograma da correlação entre cidades e churn
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Encontrar as 20 cidades com MAIS clientes no dataset
top_20_cidades = df['City'].value_counts().head(20).index

# 2. Filtrar o dataframe original apenas para essas 20 cidades
df_top_cidades = df[df['City'].isin(top_20_cidades)]

# 3. Plotar o gráfico de barras da taxa de churn para essas cidades
plt.figure(figsize=(14, 6))
sns.barplot(
    x='City', 
    y='Churn Value', 
    data=df_top_cidades, 
    errorbar=None, 
    palette='magma'
)# Informações básicas do dataset
print("=== ESTATÍSTICAS BÁSICAS ===")
print(f"Total de clientes: {len(df)}")
# Ajustes visuais para rotacionar os nomes das cidades e não encavalarem
plt.xticks(rotation=45, ha='right')
plt.title('Taxa de Churn nas 20 Cidades com Mais Clientes')
plt.ylabel('Taxa de Churn (Média)')
plt.xlabel('Cidades')
plt.ylim(0, 1)
plt.tight_layout() # Evita cortar as letras de baixo
plt.show()

